# Classification Task – Template Notebook

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score, StratifiedKFold
from scipy.stats import uniform
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, classification_report, matthews_corrcoef
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, LabelEncoder
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42


In [ ]:
def evaluate_model(X_train, X_val, y_train, y_val, pipeline):
    predict_train = pipeline.predict(X_train)
    predict_valid = pipeline.predict(X_val)

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    conf_train = confusion_matrix(y_train, predict_train)
    conf_train_norm = conf_train / np.sum(conf_train, axis=1, keepdims=True)
    sns.heatmap(conf_train_norm,
                annot=True,
                fmt='.2f',
                yticklabels=LABELS,
                xticklabels=LABELS,
                ax=axes[0],
                annot_kws={'fontsize': 12})
    axes[0].set_title('Confusion Matrix for Train Set', fontsize=14)
    axes[0].set_xlabel('Predicted label', fontsize=12)
    axes[0].set_ylabel('True label', fontsize=12)
    axes[0].tick_params(axis='x', labelsize=10)
    axes[0].tick_params(axis='y', labelsize=10)

    conf_val = confusion_matrix(y_val, predict_valid)
    conf_val_norm = conf_val / np.sum(conf_val, axis=1, keepdims=True)
    sns.heatmap(conf_val_norm,
                annot=True,
                fmt='.2f',
                yticklabels=LABELS,
                xticklabels=LABELS,
                ax=axes[1],
                annot_kws={'fontsize': 12})
    axes[1].set_title('Confusion Matrix for Validation Set', fontsize=14)
    axes[1].set_xlabel('Predicted label', fontsize=12)
    axes[1].set_ylabel('True label', fontsize=12)
    axes[1].tick_params(axis='x', labelsize=10)
    axes[1].tick_params(axis='y', labelsize=10)

    plt.tight_layout()
    plt.show()

    print('classification report for train data:')
    print(classification_report(y_train, predict_train, target_names=LABELS))
    print('classification report for validation data:')
    print(classification_report(y_val, predict_valid, target_names=LABELS))

    mcc_best = matthews_corrcoef(y_val, predict_valid)
    print(f'\nMatthews Correlation Coefficient (MCC) for Best Logistic Regression Model: {mcc_best:.4f}')


def compare_models(models_dict, X, y):
    results = []
    for name, pipeline in models_dict.items():
        y_pred = pipeline.predict(X)
        mcc = matthews_corrcoef(y, y_pred)
        results.append({"Model": name, "MCC": mcc})
    return pd.DataFrame(results).sort_values("MCC", ascending=False)


def plot_feature_importance(coefficients, pipeline):
    feature_names = pipeline.named_steps["preprocessor"].get_feature_names_out()
    feature_names = list(feature_names)

    coef_df = pd.DataFrame({
        "Feature": feature_names,
        "Coefficient": coefficients
    }).sort_values("Coefficient", key=abs, ascending=False)

    display(coef_df.head(10))
    fig = sns.barplot(
        data=coef_df,
        x='Coefficient',
        y='Feature'
    )
    # plt.title("Feature Importance", fontsize=14)
    # plt.xlabel("Coefficient Value", fontsize=12)
    plt.ylabel("Feature", fontsize=10)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.grid(True, alpha=0.3)
    return fig

In [ ]:
import scipy.stats as st

def categor_analysis(dataset, feature, name):
    '''For categorial features returns:
    plots:
    Bar plot x=feature colored by target "y"
    Boxplot
    Bar plot for target "y", y=feature

    dataset - input dataset
    feature - name of feature in dataset which is needed to analysis
    name - feature's name for plot
    '''

    plt.figure(figsize=(30, 21))
    plt.subplot(311)

    sns.countplot(x=feature, data=dataset)
    plt.xlabel(name)

    plt.subplot(312)
    sns.countplot(x=feature, hue = TARGET_COL, data=dataset)
    plt.xlabel(name)

    plt.subplot(313)
    sns.countplot(x=TARGET_COL, hue=feature, data=dataset)
    plt.show();

    cross_tab = pd.crosstab(df[feature], df[TARGET_COL])
    print("the contingency table:")
    print(cross_tab)
    stat, p, dof, expected = st.chi2_contingency(cross_tab)

    print("Chi_squared criteria between feature '{}' and 'y': P-value = {:2.4f}".format(feature, p))

In [ ]:
def plot_numerical_distribution(df, column):
    """
    Plots the histogram and a horizontal box plot for a given numerical column.
    """
    plt.figure(figsize=(12, 5))

    # Histogram
    plt.subplot(1, 2, 1)
    sns.histplot(df[column].dropna(), kde=True)
    plt.title(f'Histogram of {column}')
    plt.xlabel(column)
    plt.ylabel('Frequency')

    # Horizontal Box Plot
    plt.subplot(1, 2, 2)
    sns.boxplot(x=df[column].dropna())
    plt.title(f'Box Plot of {column}')
    plt.xlabel(column)

    plt.tight_layout()
    plt.show()

def plot_categorical_distribution(df, column):
    """
    Plots the distribution of a given categorical column using a count plot.
    """
    plt.figure(figsize=(8, 5))
    sns.countplot(data=df, x=column, palette='viridis')
    plt.title(f'Distribution of {column}')
    plt.xlabel(column)
    plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

## 1. Data Loading and task definition

In [ ]:
df = pd.read_csv('dataset_57_hypothyroid.csv')
pd.options.display.max_columns = 30
df.info()

In [ ]:
TARGET_COL = 'Class'

## 2. Exploratory Data Analysis

In [ ]:
df.info()

Let's check duplicates

In [ ]:
df.duplicated().sum()

In [ ]:
df.loc[df.duplicated(keep=False) == True]

### Univariate analysis

Let's look on categorical features

In [ ]:
categorical_cols = df.select_dtypes(include='object').columns.tolist()

for col in categorical_cols:
    plot_categorical_distribution(df, col)


and let's look on numerical

In [ ]:
numerical_cols = df.select_dtypes(include=np.number).columns.tolist()

for col in numerical_cols:
    plot_numerical_distribution(df, col)

In [ ]:
for col in df.columns:
    df.loc[df[col] == '?',col] = np.nan
for column in df.columns:
    df[column] = df[column].astype('float64', errors = 'ignore')
df = df.drop_duplicates(keep= 'first')
df = df.loc[df['age'] < 100]
df = df.drop(columns=['TBG', 'TBG_measured', 'referral_source'], axis=1)

In [ ]:
# for feature in df.select_dtypes(include=['object']):
#     categor_analysis(df, feature, feature)

In [ ]:
sns.histplot(df[TARGET_COL], kde=True)
plt.title("Target Distribution")
plt.show()


In [ ]:
df[TARGET_COL].unique()

In [ ]:
df = df.loc[df[TARGET_COL] != "secondary_hypothyroid"]

In [ ]:
LABELS = df[TARGET_COL].unique()

## 3. Train / Test Split

In [ ]:
target = df[TARGET_COL].values

encode = LabelEncoder()
target_le = encode.fit_transform(target)

keys = encode.classes_
values = encode.transform(encode.classes_)
dictionary = dict(zip(keys, values))
print(dictionary)

In [ ]:
X = df.drop(columns=[TARGET_COL])
y = df[TARGET_COL]

print(f"Shape of data_X: {X.shape}")
print(f"Shape of data_y: {y.shape}")

# Split data into training and testing sets with stratification and shuffling
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42, # for reproducibility
    shuffle=True,
    stratify=y # ensure target distribution is similar in train and test sets
)

print(f"Shape of X_train: {X_train.shape}")
print(f"Shape of X_test: {X_test.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of y_test: {y_test.shape}")

print('Train/Test split complete.')


## 4. Preprocessing

In [ ]:
# --------------------------------
# 1. Feature groups
# --------------------------------
num_features = X.select_dtypes(include=["int64", "float64"]).columns
cat_features = X.select_dtypes(include=["object", "category", "bool"]).columns
print("Numeric features: ", num_features)
print("Categorical features: ", cat_features)

# --------------------------------
# 2. Missing value handling
# --------------------------------
# Options for numeric features:
# - strategy="mean"
# - strategy="median"
# - strategy="most_frequent"
numeric_imputer = SimpleImputer(strategy="median")

# Options for categorical features:
# - strategy="most_frequent"
# - strategy="constant" (e.g., "missing")
categorical_imputer = SimpleImputer(strategy="most_frequent")

# --------------------------------
# 3. Numeric preprocessing
# --------------------------------
# Choose ONE scaler:
# - StandardScaler()  -> zero mean, unit variance (linear models)
# - MinMaxScaler()    -> bounded range [0, 1]
# numeric_scaler = StandardScaler()
numeric_scaler = MinMaxScaler()

numeric_pipeline = Pipeline(steps=[
    ("imputer", numeric_imputer),
    ("scaler", numeric_scaler)
])

# --------------------------------
# 4. Categorical preprocessing
# --------------------------------
# Encoding options:
# - OneHotEncoder -> linear models, no ordering assumption
# - OrdinalEncoder -> tree-based models or known ordering

categorical_encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False
)

# categorical_encoder = OrdinalEncoder(
#     handle_unknown="use_encoded_value",
#     unknown_value=-1
# )

categorical_pipeline = Pipeline(steps=[
    ("imputer", categorical_imputer),
    ("encoder", categorical_encoder)
])

# --------------------------------
# 5. Column transformer
# --------------------------------
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, num_features),
        ("cat", categorical_pipeline, cat_features)
    ],
    remainder="drop"  # or "passthrough"
)

# --------------------------------
# 6. (Optional) Interpolation for time series
# --------------------------------
# NOTE: interpolation is NOT part of sklearn pipelines.
# Apply it BEFORE train-test split to avoid leakage.

# Example:
# df[num_features] = df[num_features].interpolate(
#     method="linear",
#     limit_direction="both"
# )

## 5. Logistic Regression - Baseline

In [ ]:
logreg_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(C=1, solver='lbfgs', multi_class='multinomial', random_state=42))
])
logreg_pipeline.fit(X_train, y_train)

In [ ]:
evaluate_model(X_train, X_test, y_train, y_test, logreg_pipeline)

Let's look on feature importance

In [ ]:
fig = plot_feature_importance(logreg_pipeline.named_steps["model"].coef_[0], logreg_pipeline)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Logistic Regression", fontsize=14);

### with Hyper-parameters tuning

In [ ]:
param_dist = {
    'model__C': uniform(loc=0, scale=4),
    'model__solver': [ 'lbfgs', 'saga'],
    'model__penalty': ['l1', 'l2'] # Add penalty to match solver choices
}

# 2. Initialize StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 3. Create a RandomizedSearchCV object
logreg_random_search = RandomizedSearchCV(
    estimator=logreg_pipeline,
    param_distributions=param_dist,
    n_iter=50, # Number of parameter settings that are sampled
    cv=skf,
    scoring='matthews_corrcoef',
    random_state=42,
    n_jobs=-1, # Use all available CPU cores
    verbose=1
)

logreg_random_search.fit(X_train, y_train)
best_logreg = logreg_random_search.best_estimator_

In [ ]:
evaluate_model(X_train, X_test, y_train, y_test, best_logreg)

In [ ]:
fig = plot_feature_importance(best_logreg.named_steps["model"].coef_[0], best_logreg)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Best Logistic Regression", fontsize=14);

In [ ]:
models_to_compare = {"Linear Regression": logreg_pipeline, "Best Logistic Regression": best_logreg}
comparison_df = compare_models(models_to_compare, X_test, y_test)
comparison_df

## 6. Random Forest - Baseline

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

rf_pipeline.fit(X_train, y_train)


In [ ]:
evaluate_model(X_train, X_test, y_train, y_test, rf_pipeline)

In [ ]:
fig = plot_feature_importance(rf_pipeline.named_steps["model"].feature_importances_, rf_pipeline)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Best Logistic Regression", fontsize=14);

In [ ]:
from scipy.stats import randint

print('Performing Hyperparameter Tuning for Random Forest...')

# 1. Define a parameter distribution dictionary named param_dist_rf
param_dist_rf = {
    'model__n_estimators': randint(50, 200),
    'model__max_features': ['sqrt', 'log2', None],
    'model__max_depth': randint(5, 20),
    'model__min_samples_split': randint(2, 10),
    'model__min_samples_leaf': randint(1, 5),
    'model__bootstrap': [True, False]
}

# The skf object (StratifiedKFold) was already initialized in a previous step

# 2. Create a RandomizedSearchCV object named rf_random_search
rf_random_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=param_dist_rf,
    n_iter=20,
    cv=skf,
    scoring='matthews_corrcoef',
    random_state=42,
    n_jobs=-1, # Use all available CPU cores
    verbose=1
)

# 3. Fit the rf_random_search object to the training data
rf_random_search.fit(X_train, y_train)
best_rf_model = rf_random_search.best_estimator_

In [ ]:
evaluate_model(X_train, X_test, y_train, y_test, best_rf_model)

In [ ]:
fig = plot_feature_importance(best_rf_model.named_steps["model"].feature_importances_, best_rf_model)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Best Logistic Regression", fontsize=14);

In [ ]:
models_to_compare = {"Random Forest": rf_pipeline, "Best Random Forest": best_rf_model}
comparison_df = compare_models(models_to_compare, X_test, y_test)
comparison_df

## Oversampling

In [ ]:
print('Creating and fitting a Logistic Regression pipeline with SMOTE...')

# Create the imblearn pipeline with SMOTE
rf_smote_pipeline = ImbPipeline([
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=RANDOM_STATE)),
    ('model', RandomForestClassifier(random_state=42))
])

# Fit the pipeline on the original (unresampled) training data
rf_smote_pipeline.fit(X_train, y_train)
print('Pipeline with SMOTE fitted successfully.')

In [ ]:
evaluate_model(X_train, X_test, y_train, y_test, rf_smote_pipeline)

In [ ]:
fig = plot_feature_importance(rf_smote_pipeline.named_steps["model"].feature_importances_, rf_smote_pipeline)
plt.xlabel("Coefficient Value")
plt.title("Feature Importance for Logistic Regression (SMOTE in Pipeline)", fontsize=14);

# Comparison and Conclusion

In [ ]:
models_to_compare = {"Best Logistic Regression": best_logreg, "Best Random Forest": best_rf_model, "SMOTE": rf_smote_pipeline}
comparison_df = compare_models(models_to_compare, X_test, y_test)
comparison_df